# Models

> Models for the SoilspecDL project

In [1]:
#| default_exp models

In [5]:
#| export
from functools import partial
import torch
from torch.nn import (Module, AvgPool1d, LeakyReLU,
                      AdaptiveAvgPool1d, Sequential, 
                      Dropout, Linear)

from fastai.layers import ConvLayer
from typing import Callable

In [7]:
#| export
class ConvBlockBaseline(Module):    
    def __init__(self, ni, nf):
        super().__init__()
        self.conv = Sequential(
            torch.nn.Conv1d(ni, nf, kernel_size=3, stride=1, padding=1, bias=False),
            torch.nn.BatchNorm1d(nf),
            torch.nn.LeakyReLU(negative_slope=0.1)
        )
        self.pool = torch.nn.AvgPool1d(3)
    
    def forward(self, x): return self.pool(self.conv(x))

In [8]:
#| eval: false
x = torch.randn(32, 61, 64)
block = ConvBlockBaseline(61, 128)
out = block(x)
out.shape

torch.Size([32, 128, 21])

In [9]:
#| export
class ConvBlock(Module):    
    "Convolutional block with pooling."
    def __init__(self, 
                 ni, # input channels
                 nf, # output channels
                 ks=3, # kernel size
                 stride=1, # stride
                 act_cls=partial(LeakyReLU, negative_slope=0.1)
                 ):
        super().__init__()
        self.conv = ConvLayer(ni, nf, ks=ks, stride=stride, bias=False, 
                              # norm_type='NormType.Instance',
                            #   norm_type='NormType.Batch',
                              act_cls=act_cls,
                              ndim=1)
        self.pool = AvgPool1d(3)
    
    def forward(self, x): return self.pool(self.conv(x))

For instance:

In [10]:
#| eval: false
x = torch.randn(32, 61, 64)
block = ConvBlock(61, 128)
out = block(x)
out.shape

torch.Size([32, 128, 21])

In [11]:
#| export
class FeatureExtractor(Module):
    def __init__(self, 
                 in_channel=1, # input channels
                 out_channel=16, # base number of filters
                 conv_block_cls=ConvBlock,
                 ):
        super().__init__()
        nfs = [out_channel * 2**i for i in range(5)]
        self.layers = Sequential(*[
            conv_block_cls(ni=in_channel if i==0 else nfs[i-1], nf=nfs[i]) 
            for i in range(len(nfs))
        ])
        self.adaptive_pool = AdaptiveAvgPool1d(1)
    
    def forward(self, x): 
        x = self.layers(x)
        return self.adaptive_pool(x)

For instance:

In [12]:
#| eval: false
fe = FeatureExtractor(in_channel=61, out_channel=16)
x = torch.randn(32, 61, 243)
out = fe(x)
out.shape

torch.Size([32, 256, 1])

In [13]:
#| export
class MirzaiCNN(Module):
    """
    1D Convolutional Neural Network for spectral data regression/classification.
    
    Based on the architecture proposed in:
    Albinet, F., Peng, Y., Eguchi, T., Smolders, E., Dercon, G., 2022. Prediction of exchangeable potassium in soil through mid-infrared spectroscopy and deep learning: From prediction to explainability. Artificial Intelligence in Agriculture 6, 230–241. https://doi.org/10.1016/j.aiia.2022.10.001
    """
    def __init__(self, 
                 in_channel: int=1, # input channels
                 out_channel: int=16, # base number of filters
                 conv_block_cls: Callable = ConvBlock,
                 is_classifier: bool=False, # if True, model will output probabilities
                 dropout: float=0.4 # dropout rate
                 ):
        super().__init__()
        self.backbone = FeatureExtractor(in_channel, out_channel, conv_block_cls)
        self.head = Sequential(
            Dropout(dropout), 
            Linear(256, 1))
        self.is_classifier = is_classifier
        
    def forward(self, x):
        x = self.backbone(x)
        x = x.squeeze(-1)
        x = self.head(x)
        x = x.squeeze(-1)
        return torch.sigmoid(x) if self.is_classifier else x

For instance:

In [14]:
#| eval: false
model_reg = MirzaiCNN(in_channel=1, out_channel=16)
model_cls = MirzaiCNN(in_channel=1, out_channel=16, is_classifier=True)

x = torch.randn(32, 1, 243)
pred_reg = model_reg(x)
pred_cls = model_cls(x)

pred_reg.shape, pred_cls.shape, pred_cls.min().item(), pred_cls.max().item()

(torch.Size([32]), torch.Size([32]), 0.22773000597953796, 0.5989965796470642)

In [15]:
#| export
def cnt_params(model):
    for name, param in model.named_parameters():
        print(f"{name}: {param.numel():,} parameters")
    print(f"\nTotal: {sum(p.numel() for p in model.parameters()):,} parameters")

In [16]:
#| eval: False
cnt_params(model_reg)

backbone.layers.0.conv.0.weight: 48 parameters
backbone.layers.0.conv.1.weight: 16 parameters
backbone.layers.0.conv.1.bias: 16 parameters
backbone.layers.1.conv.0.weight: 1,536 parameters
backbone.layers.1.conv.1.weight: 32 parameters
backbone.layers.1.conv.1.bias: 32 parameters
backbone.layers.2.conv.0.weight: 6,144 parameters
backbone.layers.2.conv.1.weight: 64 parameters
backbone.layers.2.conv.1.bias: 64 parameters
backbone.layers.3.conv.0.weight: 24,576 parameters
backbone.layers.3.conv.1.weight: 128 parameters
backbone.layers.3.conv.1.bias: 128 parameters
backbone.layers.4.conv.0.weight: 98,304 parameters
backbone.layers.4.conv.1.weight: 256 parameters
backbone.layers.4.conv.1.bias: 256 parameters
head.1.weight: 256 parameters
head.1.bias: 1 parameters

Total: 131,857 parameters
